# 03 - Embeddings Tradicionais e Comparacao

Neste notebook geramos embeddings com `HOG`, visualizamos os resultados em 2D e salvamos a configuracao padrao do extrator considerando **as imagens originais e as imagens sinteticas**.


In [1]:
import pickle
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from math import ceil
from skimage.color import rgb2gray
from skimage.feature import hog
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import umap
except ImportError:
    umap = None

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_ROOT = Path('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')
SYNTHETIC_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'stylegan2_synthetic_256'
HOG_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_plus_synthetic_hog.parquet'
DEEP_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_plus_synthetic_deep_resnet50.parquet'
MODEL_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'hog_extractor_config.pkl'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
SAMPLE_FRACTION = 0.15
SAMPLE_RANDOM_SEED = 42
MAX_SAMPLES_PER_SOURCE = None
IMAGE_SIZE = 256
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)

for folder in [HOG_OUTPUT_PATH.parent, MODEL_OUTPUT_PATH.parent, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Originais em:', DATASET_ROOT)
print('Sinteticas em:', SYNTHETIC_ROOT)
print('Fracao da amostra:', SAMPLE_FRACTION)
print('Sample random seed:', SAMPLE_RANDOM_SEED)
print('Maximo por fonte:', MAX_SAMPLES_PER_SOURCE)
print('Tamanho da imagem para HOG:', IMAGE_SIZE)


Originais em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba
Sinteticas em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/processed/stylegan2_synthetic_256
Fracao da amostra: 0.15
Sample random seed: 42
Maximo por fonte: None
Tamanho da imagem para HOG: 256


In [3]:
def build_combined_manifest(original_root, synthetic_root=None, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None):
    sources = [
        ('original', Path(original_root), '*.jpg'),
        ('synthetic', Path(synthetic_root), '*.png'),
    ]

    frames = []
    for source_name, root, pattern in sources:
        if root is None or not root.exists():
            continue
        images = sorted(root.glob(pattern))
        if images:
            sample_size = min(len(images), ceil(len(images) * sample_fraction))
            rng = random.Random(sample_random_seed)
            images = sorted(rng.sample(images, sample_size))
        if max_samples_per_source:
            images = images[:max_samples_per_source]
        if images:
            frames.append(pd.DataFrame({
                'image_name': [path.name for path in images],
                'image_path': [str(path) for path in images],
                'source': source_name,
            }))

    if not frames:
        raise FileNotFoundError('Nenhuma imagem original ou sintetica foi encontrada.')

    return pd.concat(frames, ignore_index=True)

def extract_hog_embeddings(original_root, synthetic_root, output_path, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None, image_size=96, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    manifest = build_combined_manifest(
        original_root=original_root,
        synthetic_root=synthetic_root,
        sample_fraction=sample_fraction,
        sample_random_seed=sample_random_seed,
        max_samples_per_source=max_samples_per_source,
    )
    print(f'Extraindo HOG para {len(manifest)} imagens em {image_size}x{image_size}...')
    metadata_rows = []
    feature_rows = []
    for index, row in enumerate(manifest.itertuples(index=False), start=1):
        with Image.open(row.image_path) as image:
            image = image.convert('RGB').resize((image_size, image_size))
            gray = rgb2gray(image)
        embedding = hog(gray, pixels_per_cell=pixels_per_cell, cells_per_block=cells_per_block, feature_vector=True).astype(np.float32)
        metadata_rows.append({'image_name': row.image_name, 'image_path': row.image_path, 'source': row.source})
        feature_rows.append(embedding)
        if index % 250 == 0 or index == len(manifest):
            print(f'  processadas {index}/{len(manifest)} imagens')
    feature_columns = [f'f_{index:04d}' for index in range(feature_rows[0].shape[0])]
    metadata_frame = pd.DataFrame(metadata_rows)
    feature_frame = pd.DataFrame(np.vstack(feature_rows), columns=feature_columns, dtype='float32')
    frame = pd.concat([metadata_frame, feature_frame], axis=1)
    frame.to_parquet(output_path, index=False)
    return frame

def feature_matrix(frame):
    return frame.drop(columns=['image_name', 'image_path', 'source'])

def evaluate_source_classification(frame, test_size=0.20, random_state=42):
    features = feature_matrix(frame)
    labels = (frame['source'] == 'synthetic').astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        features,
        labels,
        test_size=test_size,
        random_state=random_state,
        stratify=labels,
    )
    estimator = Pipeline([
        ('scaler', StandardScaler(with_mean=False)),
        ('classifier', LogisticRegression(max_iter=2000, random_state=random_state)),
    ])
    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    return {
        'metric': 'source_classification',
        'accuracy': float(accuracy),
        'error_rate': float(1.0 - accuracy),
        'train_samples': int(len(X_train)),
        'test_samples': int(len(X_test)),
    }

def reduce_embeddings(frame, method='pca', random_state=42):
    features = feature_matrix(frame)
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=random_state, init='pca')
    elif method == 'umap':
        if umap is None:
            raise ImportError("Instale 'umap-learn' para usar UMAP.")
        reducer = umap.UMAP(n_components=2, random_state=random_state)
    else:
        raise ValueError(f'Metodo desconhecido: {method}')
    reduced = reducer.fit_transform(features)
    result = frame[['image_name', 'image_path', 'source']].copy()
    result['x'] = reduced[:, 0]
    result['y'] = reduced[:, 1]
    result['method'] = method
    return result

def save_scatter_plot(frame, output_path, title, hue='source'):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=frame, x='x', y='y', hue=hue, alpha=0.75, s=40)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

def save_pickle_model(model, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as file:
        pickle.dump(model, file)
    return output_path

def build_hog_extractor_config(image_size=96, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    return {
        'method': 'hog',
        'image_size': image_size,
        'pixels_per_cell': pixels_per_cell,
        'cells_per_block': cells_per_block,
    }


## Embeddings HOG

In [ ]:
hog_df = extract_hog_embeddings(
    original_root=DATASET_ROOT,
    synthetic_root=SYNTHETIC_ROOT,
    output_path=HOG_OUTPUT_PATH,
    sample_fraction=SAMPLE_FRACTION,
    sample_random_seed=SAMPLE_RANDOM_SEED,
    max_samples_per_source=MAX_SAMPLES_PER_SOURCE,
    image_size=IMAGE_SIZE,
    pixels_per_cell=HOG_PIXELS_PER_CELL,
    cells_per_block=HOG_CELLS_PER_BLOCK,
)
hog_df.head()


Extraindo HOG para 34037 imagens em 256x256...
  processadas 250/34037 imagens
  processadas 500/34037 imagens
  processadas 750/34037 imagens
  processadas 1000/34037 imagens
  processadas 1250/34037 imagens
  processadas 1500/34037 imagens
  processadas 1750/34037 imagens
  processadas 2000/34037 imagens
  processadas 2250/34037 imagens
  processadas 2500/34037 imagens
  processadas 2750/34037 imagens
  processadas 3000/34037 imagens
  processadas 3250/34037 imagens
  processadas 3500/34037 imagens
  processadas 3750/34037 imagens
  processadas 4000/34037 imagens
  processadas 4250/34037 imagens
  processadas 4500/34037 imagens
  processadas 4750/34037 imagens
  processadas 5000/34037 imagens
  processadas 5250/34037 imagens
  processadas 5500/34037 imagens
  processadas 5750/34037 imagens
  processadas 6000/34037 imagens
  processadas 6250/34037 imagens
  processadas 6500/34037 imagens
  processadas 6750/34037 imagens
  processadas 7000/34037 imagens
  processadas 7250/34037 imagens

In [ ]:
hog_2d = reduce_embeddings(hog_df, method='pca')
hog_2d.head()


,image_name,image_path,source,x,y,method
0,000107.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.167072,0.449212,pca
1,000150.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,-1.286252,0.724185,pca
2,000178.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,2.031296,-1.275252,pca
3,000285.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,-0.799844,0.756937,pca
4,000302.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.777872,-0.609630,pca


In [ ]:
figure_path = FIGURES_DIR / 'celeba_align_hog_pca.png'
save_scatter_plot(hog_2d, figure_path, 'CelebA Align - HOG (PCA)')
figure_path

PosixPath('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/reports/figures/celeba_align_hog_pca.png')

In [ ]:
hog_config = build_hog_extractor_config(
    image_size=IMAGE_SIZE,
    pixels_per_cell=HOG_PIXELS_PER_CELL,
    cells_per_block=HOG_CELLS_PER_BLOCK,
)
model_path = save_pickle_model(hog_config, MODEL_OUTPUT_PATH)
hog_metrics = evaluate_source_classification(hog_df, random_state=SAMPLE_RANDOM_SEED)
print('Configuracao padrao do HOG salva em:', model_path)
print('Accuracy (original vs synthetic):', round(hog_metrics['accuracy'], 4))
print('Error rate:', round(hog_metrics['error_rate'], 4))
pd.DataFrame([hog_metrics])

Features: (1203, 4356)
Configuracao padrao do HOG salva em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/artifacts/models/hog_extractor_config.pkl
Accuracy (original vs synthetic): 0.9959
Error rate: 0.0041


,metric,accuracy,error_rate,train_samples,test_samples
0,source_classification,0.995851,0.004149,962,241


## Comparacao entre deep e HOG

In [ ]:
if DEEP_OUTPUT_PATH.exists():
    deep_df = pd.read_parquet(DEEP_OUTPUT_PATH)
    deep_metrics = evaluate_source_classification(deep_df, random_state=SAMPLE_RANDOM_SEED)
    comparison_df = pd.DataFrame([
        {'embedding': 'deep_resnet50', **deep_metrics},
        {'embedding': 'hog', **hog_metrics},
    ])
    comparison_df[['embedding', 'accuracy', 'error_rate', 'train_samples', 'test_samples']]
else:
    print('Execute antes o notebook 03 para gerar os embeddings profundos combinando imagens originais e sinteticas.')


In [ ]:
print('Comparacao de acuracia entre HOG e ResNet50:')
if 'comparison_df' in locals():
    print(comparison_df[['embedding', 'accuracy', 'error_rate']])
else:
    print('Dados de comparacao nao disponiveis. Execute o notebook 03 para gerar os embeddings profundos combinando imagens originais e sinteticas.')

Comparacao de acuracia entre HOG e ResNet50:
       embedding  accuracy  error_rate
0  deep_resnet50  1.000000    0.000000
1            hog  0.995851    0.004149
